In [ ]:
import tiktoken

In [ ]:
text = """At midnight, Maya heard a soft knock beneath her bed.
She froze, then slowly looked down. A tiny blue door had appeared in the floorboards.
Curious, she opened it and discovered a glowing forest filled with floating lanterns.
A fox wearing a silver bell bowed and whispered, “The stars are disappearing.”
Maya followed him to a crystal lake, where the final star trembled above the water.
She reached out and caught it gently. Instantly, thousands of stars burst across the sky.
The fox smiled. “You saved the night.” Maya returned home before sunrise, carrying one tiny silver bell in her pocket.
"""

In [ ]:
encoder = tiktoken.encoding_for_model('gpt-3.5-turbo')

In [ ]:
tokens = encoder.encode(text)

In [ ]:
len(tokens)

133

In [ ]:
tokens

[1688,
 33433,
 11,
 51444,
 6755,
 264,
 8579,
 14459,
 24923,
 1077,
 4950,
 13,
 720,
 8100,
 90109,
 11,
 1243,
 14297,
 7111,
 1523,
 13,
 362,
 13987,
 6437,
 6134,
 1047,
 9922,
 304,
 279,
 6558,
 19826,
 13,
 720,
 17119,
 1245,
 11,
 1364,
 9107,
 433,
 323,
 11352,
 264,
 49592,
 13952,
 10409,
 449,
 19596,
 74265,
 82,
 13,
 720,
 32,
 39935,
 12512,
 264,
 15310,
 29519,
 85473,
 323,
 58366,
 11,
 1054,
 791,
 9958,
 527,
 67503,
 2029,
 720,
 11356,
 64,
 8272,
 1461,
 311,
 264,
 26110,
 22553,
 11,
 1405,
 279,
 1620,
 6917,
 18659,
 38759,
 3485,
 279,
 3090,
 13,
 720,
 8100,
 8813,
 704,
 323,
 10791,
 433,
 30373,
 13,
 18549,
 398,
 11,
 9214,
 315,
 9958,
 21165,
 4028,
 279,
 13180,
 13,
 720,
 791,
 39935,
 31645,
 13,
 1054,
 2675,
 6924,
 279,
 3814,
 2029,
 51444,
 6052,
 2162,
 1603,
 64919,
 11,
 15691,
 832,
 13987,
 15310,
 29519,
 304,
 1077,
 18301,
 627]

In [2]:
from openai import OpenAI
client = OpenAI(
    api_key='voc-3231998021403751658726a2fc7d2440498.60015628',
    base_url='https://openai.vocareum.com/v1',
)

In [3]:
response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant. respond to the user as per the question in a gentle way"},
                {"role": "user", "content": text}
            ],
            temperature=0.2,
            timeout=60
            )

NameError: name 'text' is not defined

In [24]:
from pydantic import BaseModel, Field

class School(BaseModel):
  name : str = Field(description="Name of the School.")
  adress : str = Field(description="None")

class APIInput(BaseModel):
  name : str = Field(description="Name of the student.")
  age : int = Field(description="Age of the student.", ge = 17, le=120)
  mark : list[int] = Field(description="Marks of the student.")
  description : str = Field(description="description of the student.", min_lenght = 5, max_length = 100)
  school : School

/tmp/ipykernel_941/3474411588.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'min_lenght'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  description : str = Field(description="description of the student.", min_lenght = 5, max_length = 100)


In [26]:
student = {
    "name" : "John",
    "age" : 135,
    "mark" : [30,40,50],
    "description" : "The fnpewnfpw fwefjpowejf fewjfeojfw",
    "raw_out" : ""
}

In [27]:
try:
  APIInput.model_validate(student)
except Exception as e:
  print(f"Call LLM... along with {e}")
  print(e)

Call LLM... along with 2 validation errors for APIInput
age
  Input should be less than or equal to 120 [type=less_than_equal, input_value=135, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
mark
  Field required [type=missing, input_value={'name': 'John', 'age': 1...jfeojfw', 'raw_out': ''}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
2 validation errors for APIInput
age
  Input should be less than or equal to 120 [type=less_than_equal, input_value=135, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
mark
  Field required [type=missing, input_value={'name': 'John', 'age': 1...jfeojfw', 'raw_out': ''}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


# **Pydantic + Parsing**

In [9]:
from pydantic import BaseModel, Field, ValidationError
from typing import List
import json

retries = 3

class Answer(BaseModel):
  content: str = Field(description="The main answer content.")
  confidence: float = Field(description="Confidence score between 0.0 and 1.0.")
  sources: List[str] = Field(description="List of sources or references used.")

def call_llm(error_msg):
  # there is an error in the prompt insted of filed 'sources' (as per pydantic) model I provided source in prompt to generate an pydantic vaidation error)
  message = [{"role" : "system", "content" :  f"""Anser the user question in the JSON format which contains keys like content, confidance (Score bw 0-1), sources (Refference source)
  error from previous call= {error_msg}
  sample output if there nay error from previous call rectify it
  {{
  content: <the real response>,
  confidence : <the confidance socre range bw 0-1>,
  source: [<source1>,<sorce2>]
  }}
  """},
           {"role" : "user", "content" :  "How is the growth rate of India ?"}]


  response = client.chat.completions.create(
    model = 'gpt-3.5-turbo',
    messages=message,
  )
ans = None
error_msg = None
for i in range(0,retries):
  try:
    response = call_llm(error_msg)
    output = response.choices[0].message.content
    ans = Answer.model_validate(output)
    break
  except ValidationError as e:
    print("Error appeared and trying Again")
    error_msg = e
    continue

print(ans)

None


NameError: name 'output' is not defined

# **LLM Json Mode**

In [ ]:
from pydantic import BaseModel, Field
from typing import List
import json

class Answer(BaseModel):
  content: str = Field(description="The main answer content.")
  confidence: float = Field(description="Confidence score between 0.0 and 1.0.")
  sources: List[str] = Field(description="List of sources or references used.")

message = [{"role" : "system", "content" :  """Anser the user question in the JSON format which contains keys like content, confidance (Score bw 0-1), sources (Refference source)
sample output
{
 content: <the real response>,
 confidence : <the confidance socre range bw 0-1>,
 sources: [<source1>,<sorce2>]
}
"""},
           {"role" : "user", "content" :  "How is the growth rate of India ?"}]


response = client.chat.completions.create(
  model = 'gpt-3.5-turbo',
  messages=message,
  response_format = {"type":"json_object"} # explicitly commanding LLM to respond n JSON
)

output = response.choices[0].message.content
print(output)
json_output = json.loads(output)

ans = Answer.model_validate(json_output)

print(ans)

{
    "content": "As of 2021, the growth rate of India is projected to be around 8-9% in the coming years.",
    "confidence": 0.85,
    "sources": ["https://www.worldbank.org/en/country/india/overview"]
}
content='As of 2021, the growth rate of India is projected to be around 8-9% in the coming years.' confidence=0.85 sources=['https://www.worldbank.org/en/country/india/overview']


In [ ]:
tool = [
    {
        "type": "function",
        "function": {
            "name": "answer_question",
            "description": "Provide a structured answer to the user's question",
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {
                        "type": "string",
                        "description": "The main answer content."
                    },
                    "confidence": {
                        "type": "number",
                        "description": "Confidence score between 0.0 and 1.0."
                    },
                    "sources": {
                        "type": "array",
                        "description": "List of sources or references used",
                        "items": {
                            "type": "string"
                        }
                    }
                },

                "required": ["content", "confidence", "sources"]
            }
        }
    }
]


message = [{"role" : "user", "content" :  "How is the growth rate of India ?"}]


response = client.chat.completions.create(
  max_token = 1024,
  model = 'gpt-3.5-turbo',
  messages=message,
  tools=tool,
  tool_choice={"type": "function", "function": {"name": "answer_question"}}, # Enforce the usage of this tool in every LLM call
)

# response.choices[0].message.content - LLM response
tool_call = response.choices[0].message.tool_calls[0] # LLM Tool call reponse
output = tool_call.function.arguments
print(output)
json_output = json.loads(output)

ans = Answer.model_validate(json_output)

print(ans.content)

{"content":"India has experienced varying growth rates over the years. As of the most recent data, the growth rate of India's GDP was around 4.2% in 2019. It is essential to note that economic growth rates can fluctuate due to various factors such as government policies, global economic conditions, and domestic economic activities. For the latest and most accurate information, it is recommended to refer to official government reports and data sources.","confidence":0.9,"sources":[]}
India has experienced varying growth rates over the years. As of the most recent data, the growth rate of India's GDP was around 4.2% in 2019. It is essential to note that economic growth rates can fluctuate due to various factors such as government policies, global economic conditions, and domestic economic activities. For the latest and most accurate information, it is recommended to refer to official government reports and data sources.


In [ ]:
print(ans.content)

India has experienced varying growth rates over the years. As of the most recent data, the growth rate of India's GDP was around 4.2% in 2019. It is essential to note that economic growth rates can fluctuate due to various factors such as government policies, global economic conditions, and domestic economic activities. For the latest and most accurate information, it is recommended to refer to official government reports and data sources.


In [ ]:
ans.confidence

0.9

In [ ]:
ans.sources

[]

In [ ]:
tools_calls = response.choices[0].message.tool_calls

In [ ]:
tool_name = "answer_question"
for tool_call in tools_calls:
  if tool_call.function.name == tool_name:
    print(tool_call.function.arguments)

{"content":"India has experienced varying growth rates over the years. As of the most recent data, the growth rate of India's GDP was around 4.2% in 2019. It is essential to note that economic growth rates can fluctuate due to various factors such as government policies, global economic conditions, and domestic economic activities. For the latest and most accurate information, it is recommended to refer to official government reports and data sources.","confidence":0.9,"sources":[]}


In [ ]:
# tool 1
# tool 2
# tool 3

# base iput eiter tool 1 or tool 2
# tool 3 should always get invok